<a href="https://colab.research.google.com/github/Lucaschewitch/Machine-Learning/blob/main/%D0%9F%D0%A012(2(%D0%BD%D0%B5%D0%B7%D0%B0%D0%BA%D0%BE%D0%BD%D1%87).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа. Геомаркетинговое исследование территорий с применением методов машинного обучения**


## **Цель работы**


Овладеть методами пространственного анализа и машинного обучения для решения практической задачи определения оптимальных локаций размещения коммерческих объектов.

## **Введение**


В современном бизнесе местоположение коммерческого объекта играет ключевую роль в его успешности. Геомаркетинговый анализ позволяет объективно оценить привлекательность различных локаций, опираясь на количественные показатели и алгоритмы машинного обучения.

В рамках данной работы вы примените полный цикл пространственного анализа, включая сбор данных из открытых источников, обработку и агрегацию пространственной информации, обучение моделей машинного обучения и визуализацию результатов.

## **Задание**


Провести геомаркетинговое исследование для выбора оптимальных локаций размещения новых точек определенного типа бизнеса на территории выбранного города.

## **Порядок выполнения работы**

### **Часть 1. Подготовка данных и определение задачи**



1. **Индивидуальный выбор территории и типа бизнеса**:
   - Выберите город/район для проведения анализа (например, центральная часть Москвы, Санкт-Петербурга или любого другого крупного города)

   **Ответ**: выберем город Подольск. Он достаточно большой для анализа, но недостаточно, чтобы возникли проблемы с вычислительной мощностью.
   - Определите тип бизнеса для анализа (аптеки, продуктовые магазины, пункты выдачи заказов, рестораны определенной кухни и т.д.)

   **Ответ**: Аптеки. Социально значимый бизнес, легко подобрать характеристики для выбора оптимальной локации.
   - Обоснуйте свой выбор: почему данная территория и тип бизнеса интересны для анализа?
   **Ответ**: Подольск - динамично развивающийся город с густой застройкой и отличной транспортной доступностью. Подольчанам определённо нужны точки для покупки лекарственных препаратов.

2. **Определение ключевых факторов успешности**:

   Самостоятельно сформулируйте не менее 5 факторов, которые могут влиять на успешность выбранного типа бизнеса

   **Ответ**:
    - Населённость. Количество жилых зданий в сетке территории.
    - Транспортная доступность. Наличие остановок общественного транспорта и станций.
    - Близость к медицинским учреждениям. Количество поликлиник и больниц рядом.
    - Уровень конкуренции. Наличие существующих аптек в радиусе.
    - Потоковый трафик. Количество магазинов или ТЦ, генерирующих пешеходный поток.

   Для каждого фактора определите, какими данными из OpenStreetMap его можно количественно описать

   **Ответ**:
   - Населённость: building=residential, building=apartments.
   - Транспортная доступность: highway=bus_stop railway=station, public_transport=stop_area.
   - Близость к мед. учреждениям: amenity=clinic, amenity=hospital.
   - Конкуренция: amenity=pharmacy.
   - Проходимость: shop=supermarket, shop=mall.

   Составьте таблицу соответствия между факторами и тегами OpenStreetMap:

| Фактор успешности | Теги OpenStreetMap |
| :--- | :--- |
| Плотность населения | `building=residential`, `building=apartments` |
| Транспортная доступность | `highway=bus_stop`, `railway=station`, `public_transport=stop_area` |
| Близость к мед. учреждениям | `amenity=clinic`, `amenity=hospital` |
| Уровень конкуренции | `amenity=pharmacy` |
| Потоковый трафик (магазины/ТЦ) | `shop=supermarket`, `shop=mall`, `shop=convenience` |

3. **Сбор исходных данных**:
   - Настройте необходимые библиотеки из теоретического материала
   - Определите необходимую область интереса (ROI) с помощью интерактивной карты
   - Загрузите данные о существующих объектах вашего типа бизнеса и объектах инфраструктуры, связанных с выделенными вами факторами

In [26]:
!pip install osmnx geopandas leafmap mapclassify h3pandas h3~=3.0
!apt install -y libspatialindex-dev
!pip install geopandas osmnx folium shapely matplotlib numpy pandas scikit-learn h3==3.7.6
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from shapely.geometry import Point, Polygon
import time
import h3
import h3pandas
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

ox.settings.use_cache = True
ox.settings.log_console = False
ox.settings.overpass_endpoint = "https://overpass.kumi.systems/api/interpreter"
ox.settings.timeout = 300

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libspatialindex-c6 libspatialindex6
The following NEW packages will be installed:
  libspatialindex-c6 libspatialindex-dev libspatialindex6
0 upgraded, 3 newly installed, 0 to remove and 3 not upgraded.
Need to get 319 kB of archives.
After this operation, 1,416 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libspatialindex6 amd64 1.9.3-2 [247 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libspatialindex-c6 amd64 1.9.3-2 [55.8 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libspatialindex-dev amd64 1.9.3-2 [16.0 kB]
Fetched 319 kB in 0s (4,607 kB/s)
Selecting previously unselected package libspatialindex6:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libspatialindex6_1.9.3-2_amd64.deb ...
Un

In [15]:
g = ox.geocode_to_gdf("Подольск, Россия")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [16]:
p = "Подольск"
t = {"building": ["residential", "apartments"]}
bld = ox.features_from_place(p, t)

/usr/local/lib/python3.12/dist-packages/osmnx/features.py:292: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon = gdf_place["geometry"].unary_union
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [17]:
t = {"amenity": "pharmacy"}
ph = ox.features_from_place(p, t)

/usr/local/lib/python3.12/dist-packages/osmnx/features.py:292: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon = gdf_place["geometry"].unary_union
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [20]:
t = {"highway": "bus_stop"}
bus = ox.features_from_place(p, t)

/usr/local/lib/python3.12/dist-packages/osmnx/features.py:292: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon = gdf_place["geometry"].unary_union
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [21]:
t = {"amenity": ["clinic", "hospital"]}
med = ox.features_from_place(p, t)

/usr/local/lib/python3.12/dist-packages/osmnx/features.py:292: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon = gdf_place["geometry"].unary_union
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [22]:
t = {"shop": ["supermarket", "mall", "convenience"]}
shops = ox.features_from_place(p, t)

/usr/local/lib/python3.12/dist-packages/osmnx/features.py:292: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon = gdf_place["geometry"].unary_union
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
t = {"amenity": ["school", "university"]}
shops = ox.features_from_place(p, t)
t = {"amenity": ["bank", "atm"]}
shops = ox.features_from_place(p, t)

### **Часть 2. Пространственное агрегирование и создание признаков**


1. **Создание гексагональной сетки**:
   - Самостоятельно определите оптимальную детализацию сетки H3 для вашего анализа
   - Обоснуйте выбор разрешения (resolution) сетки с учётом масштаба вашей территории и специфики бизнеса
   - Покройте территорию гексагональной сеткой выбранного разрешения

2. **Инженерия пространственных признаков**:
   - Определите радиусы для анализа ближнего и среднего окружения (могут отличаться от предложенных в теории в зависимости от специфики вашего бизнеса)
   - Разработайте и реализуйте не менее 10 пространственных признаков, описывающих характеристики каждой ячейки
   - **Дополнительное задание**: придумайте и реализуйте не менее 2 пространственных признаков, которых нет в теоретическом материале

3. **Анализ полученных признаков**:
   - Рассчитайте базовую статистику по каждому признаку
   - Исследуйте корреляции между признаками
   - Выявите и обработайте выбросы и пропущенные значения, если они имеются

In [30]:
c = (55.42972, 37.54444)
size = 0.02
xmin, xmax = c[1] - size, c[1] + size
ymin, ymax = c[0] - size, c[0] + size
bbox = (ymin, xmin, ymax, xmax)

In [32]:
import h3
from shapely.geometry import Polygon
import geopandas as gpd

ymin, xmin, ymax, xmax = bbox

res_h3 = 9

polygon = Polygon([(xmin, ymin), (xmax, ymin), (xmax, ymax), (xmin, ymax)])

hex_ids = h3.polyfill_polygon([(xmin, ymin), (xmax, ymin), (xmax, ymax), (xmin, ymax)], res_h3)

hex_geom = []
for hid in hex_ids:
    boundary = h3.h3_to_geo_boundary(hid, geo_json=True)
    hex_geom.append(Polygon(boundary))

hex_gdf = gpd.GeoDataFrame({'geometry': hex_geom}, crs='EPSG:4326')
hex_gdf = hex_gdf[hex_gdf.geometry.intersects(polygon)]
hex_gdf['h3_id'] = list(hex_ids)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [39]:
def count_in_hex(gdf_obj, hex_gdf):
    if gdf_obj is None or len(gdf_obj) == 0:
        return pd.Series([0] * len(hex_gdf))
    return hex_gdf.geometry.apply(lambda x: len(gdf_obj[gdf_obj.intersects(x)]))

In [41]:
hex_gdf['n_bld'] = count_in_hex(bld, hex_gdf)
hex_gdf['n_ph'] = count_in_hex(ph, hex_gdf)
hex_gdf['n_bus'] = count_in_hex(bus, hex_gdf)
hex_gdf['n_med'] = count_in_hex(med, hex_gdf)
hex_gdf['n_shops'] = count_in_hex(shops, hex_gdf)
hex_gdf['n_schools'] = count_in_hex(schools, hex_gdf)
hex_gdf['n_banks'] = count_in_hex(banks, hex_gdf)
hex_gdf['dist_center'] = hex_gdf.geometry.centroid.distance(Point(c[1], c[0]))
hex_gdf['diversity'] = hex_gdf[['n_bus','n_med','n_shops','n_schools','n_banks']].sum(axis=1) / (hex_gdf['n_bld'] + 1)

TypeError: (<class 'geopandas.geoseries.GeoSeries'>, <class 'NoneType'>)

### **Часть 3. Моделирование привлекательности локаций**



1. **Подготовка целевой переменной**:
   - Определите, как будет сформирована целевая переменная для вашей задачи (по умолчанию: наличие объектов выбранного типа в ячейке)
   - Исследуйте распределение целевой переменной и оцените её сбалансированность
   - При необходимости, предложите стратегию работы с несбалансированными данными

2. **Разработка моделей машинного обучения**:
   - Реализуйте и обучите несколько моделей (минимум 2) для предсказания привлекательности локации
   - Проведите оценку важности признаков для каждой модели
   - Сравните модели по метрикам качества и выберите наилучшую
   - **Дополнительное задание**: настройте гиперпараметры модели с помощью поиска по сетке (GridSearchCV) или случайного поиска (RandomSearchCV)

3. **Улучшение модели с помощью кластеризации**:
   - Выполните кластеризацию ячеек по их характеристикам
   - Определите оптимальное число кластеров с помощью метода локтя или силуэта
   - Визуализируйте результаты кластеризации
   - Проверьте, улучшает ли добавление информации о кластерах качество основной модели

### **Часть 4. Расчет потенциала локаций и финальные рекомендации**



1. **Разработка интегрального показателя потенциала**:
   - Самостоятельно определите веса для факторов привлекательности среды и конкуренции
   - Обоснуйте выбранные веса в контексте вашего бизнеса
   - Рассчитайте итоговый потенциал для всех ячеек сетки
   - Категоризируйте потенциал для упрощения интерпретации результатов

2. **Визуализация результатов**:
   - Создайте интерактивную карту с тепловым слоем потенциала
   - Добавьте маркеры существующих объектов вашего типа бизнеса
   - Выделите топ-10 локаций с наивысшим потенциалом
   - Подготовьте отдельную карту фокуса на лучших локациях

3. **Формирование бизнес-рекомендаций**:
   - Составьте список из 5-7 конкретных локаций для размещения новых объектов
   - Для каждой рекомендуемой локации укажите:
     * Точные координаты
     * Значение потенциала
     * Ключевые характеристики локации
     * Преимущества и возможные риски размещения в данной точке
   - Подготовьте общие рекомендации по стратегии территориального развития для выбранного бизнеса

## **Рекомендации по выполнению**


1. Начните с малой территории для тестирования кода и методологии, затем расширяйте анализ.
2. Используйте инкрементальный подход: сначала реализуйте базовый функционал, затем улучшайте его.
3. Регулярно сохраняйте промежуточные результаты работы.
4. При выборе признаков опирайтесь не только на учебный материал, но и на научные статьи по геоанализу и геомаркетингу.
5. Обращайте внимание на особенности территории и уникальные характеристики выбранного бизнеса.
6. Для оценки результатов старайтесь сопоставить их с реальным расположением успешных объектов аналогичного бизнеса.